In [4]:
import pandas as pd

df = pd.read_csv('../data/raw/16OCLicitacion.csv', sep=';', encoding='latin-1')
df.shape

(28034, 46)

## Tratamiento de nulos en RegionUnidadCompra

La columna tenía 1.269 valores nulos. Investigando en el notebook de exploración,
se detectó que estos nulos correspondían a un grupo específico de instituciones,
no a datos faltantes aleatorios.

Se aplicó un mapeo manual basado en el nombre de cada institución:
- Instituciones con ubicación física única (hospitales, corporaciones municipales)
  fueron mapeadas a su región correspondiente.
- Instituciones de alcance nacional (ej. Servicio Nacional de Migraciones) se
  categorizaron como 'Nacional / Multiples regiones', ya que asignarles una
  única región sería inexacto.
- Un caso sin información suficiente se dejó como 'Sin especificar'.

In [8]:
mapeo_regiones = {
    'HOSPITAL COMPLEJO ASISTENCIAL PADRE LAS CASAS': 'Region de la Araucania',
    'SERVICIO LOCAL DE EDUCACIÓN PÚBLICA DE VALDIVIA': 'Region de Los Rios',
    'CORP MUNICIPAL PARA EL DESARROLLO SOCIAL': 'Region Metropolitana de Santiago',
    'Corporación de Desarrollo de La Reina': 'Region Metropolitana de Santiago',
    'Corporación Municipal de Desarrollo Productivo y Turismo de Molina': 'Region del Maule',
    'CORP MUNICIPAL DE RENCA': 'Region Metropolitana de Santiago',
    'Servicio Local de Educación Pública Iquique': 'Region de Tarapaca',
    'SERVICIO NACIONAL DE MIGRACIONES': 'Nacional / Multiples regiones',
    'SERVICIO NACIONAL DE PROTECCIÓN ESPECIALIZADA A LA NIÑEZ Y ADOLESCENCI': 'Nacional / Multiples regiones',
    'FUNDACION EDUCACIONAL PARA EL DESAROLLO INTEGRAL DE LA NIÑEZ': 'Sin especificar',
}

In [9]:
df['RegionUnidadCompra'] = df['RegionUnidadCompra'].fillna(df['Institucion'].map(mapeo_regiones))

In [10]:
df['RegionUnidadCompra'].isnull().sum()

np.int64(0)

In [11]:
df['FechaEnvioOC'].head()

0    2026-03-16
1    2026-02-05
2    2026-03-23
3    2026-03-23
4    2026-03-23
Name: FechaEnvioOC, dtype: str

## Conversión de FechaEnvioOC a tipo fecha

La columna venía como texto (str) en formato ISO (YYYY-MM-DD). Se convierte a
tipo datetime para poder extraer año, mes y trimestre en el análisis de
estacionalidad (Fase 3).

In [12]:
df['FechaEnvioOC'] = pd.to_datetime(df['FechaEnvioOC'])
df['FechaEnvioOC'].dtype

dtype('<M8[us]')